In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

entity ------> care_Epi_contract_id

MPB-------------

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(ten.id AS varchar(100))

In [ ]:
Updated MPB care_epi_contr_id mapping in the Care Episode notebook to use the new RDM Contracts table. The old source-derived contract key logic was replaced with a join to silver_rdm_contract using contr_src_sys_inst_id = 'MPB001' and contr_src_id = ten.id, and care_epi_contr_id is now populated from rdmc.contr_id as per the updated Monday definition.

In [ ]:
wip ---------------------------

In [ ]:
DevOps wording

Updated WIP care_epi_contr_id mapping to use the new RDM Contracts table. Replaced the old source-derived customer-name logic with a join to silver_rdm_contract using contr_src_sys_inst_id = 'WIP001' and contr_src_name = ahrd.customer_name, and now populate care_epi_contr_id from rdmc.contr_id.

Validation wording

Validated WIP contract mapping in select-based testing. Matching rows successfully returned rdmc.contr_id as care_epi_contr_id using the new RDM Contracts join.

Retained the existing silver_contract join as it is still used for other contract-derived attributes, while introducing silver_rdm_contract specifically for the new care_epi_contr_id mapping.

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
-- Join WIP contract to the new RDM Contracts table using source system instance and source contract name
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'WIP001'
   AND trim(lower(rdmc.contr_src_name)) = trim(lower(ahrd.customer_name))

In [ ]:
SELECT
    care_epi_id,
    care_epi_src_id,
    care_epi_contr_id,
    z_src_system_instance
FROM
    silver_care_episode
WHERE
    z_src_system_instance = 'WIP001'
    AND care_epi_contr_id IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    COUNT(*) AS total_count,
    SUM(CASE WHEN care_epi_contr_id IS NOT NULL THEN 1 ELSE 0 END) AS populated_count,
    SUM(CASE WHEN care_epi_contr_id IS NULL THEN 1 ELSE 0 END) AS null_count,
    COUNT(DISTINCT care_epi_contr_id) AS distinct_contract_id_count
FROM silver_care_episode
WHERE z_src_system_instance = 'WIP001';

In [ ]:
string(outputs('Run_a_query_against_a_dataset')?['body']?['error']?['pbi.error']?['details']?[0]?['detail']?['value'])

In [ ]:
-- Empty table create karaychi query
CREATE TABLE shape_on_list_test_add (
    test_source_name STRING,
    test_source_id INT
);

In [ ]:
INSERT INTO shape_on_list_test_add (test_source_name, test_source_id)
VALUES
    ('Pathway A', 1),
    ('Pathway B', 2),
    ('Pathway C', 3),
    ('Pathway D', 4);

In [ ]:
sn----------------------

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
-- Join SONE contract to the new RDM Contracts table using dynamic source system instance and source contract ID
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))

In [ ]:
SELECT
    COUNT(*) AS total_count,
    SUM(CASE WHEN care_epi_contr_id IS NOT NULL THEN 1 ELSE 0 END) AS populated_count,
    SUM(CASE WHEN care_epi_contr_id IS NULL THEN 1 ELSE 0 END) AS null_count
FROM silver_care_episode
WHERE z_src_system_id = 'SONE';